# 02 Bronze Ingestion Layer

In [12]:
# import libs
from datetime import date
from pathlib import Path
import os

import scanpy as sc
import pandas as pd
import h5py

import warnings
warnings.filterwarnings('ignore')

In [3]:
# load data
source_file = '../data/OSD-352/GLDS-352_snRNA-Seq_filtered_feature_bc_matrix.h5'
adata = sc.read_10x_h5(source_file)

print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"Source: {source_file}")

Loaded: 32,243 cells × 32,285 genes
Source: ../data/OSD-352/GLDS-352_snRNA-Seq_filtered_feature_bc_matrix.h5


In [5]:
# make gene names unique
adata.var_names_make_unique()

print('Gene names made unique')
print(f'Example genes: {adata.var_names[:5].tolist()}')

Gene names made unique
Example genes: ['Xkr4', 'Gm1992', 'Gm19938', 'Gm37381', 'Rp1']


In [6]:
# add provenance metadata to cell observation
provenance = {
    'ingest_date': date.today().strftime('%Y-%m-%d'),
    'source_file': 'GLDS-352_snRNA-Seq_filtered_feature_bc_matrix.h5',
    'dataset_id': 'OSD-352',
    'organism': 'Mus musculus',
    'tissue': 'brain',
    'technology': '10X Chromium snRNA-seq',
    'genome_build': 'mm10',
    'processing_pipeline': 'GeneLab scRNA-seq'
}

for key, value in provenance.items():
    adata.obs[key] = value

print("Provenance metadata added:")
for key, value in provenance.items():
    print(f" {key}: {value}")

Provenance metadata added:
 ingest_date: 2026-03-12
 source_file: GLDS-352_snRNA-Seq_filtered_feature_bc_matrix.h5
 dataset_id: OSD-352
 organism: Mus musculus
 tissue: brain
 technology: 10X Chromium snRNA-seq
 genome_build: mm10
 processing_pipeline: GeneLab scRNA-seq


In [8]:
# def output directory (hive-style partitioning)
output_dir = Path('../data/bronze/osd352_brain') / f"ingest_date={provenance['ingest_date']}"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Directory exists: {output_dir.exists()}")

Output directory: ../data/bronze/osd352_brain/ingest_date=2026-03-12
Directory exists: True


In [9]:
# save cell metadata as Parquet
obs_path = output_dir / 'obs.parquet'
adata.obs.to_parquet(obs_path)

print(f'Saved cell metadata: {obs_path}')
print(f'Shape: {adata.obs.shape}')


Saved cell metadata: ../data/bronze/osd352_brain/ingest_date=2026-03-12/obs.parquet
Shape: (32243, 8)


In [10]:
# save gene metadata as Parquet
var_path = output_dir / 'var.parquet'
adata.var.to_parquet(var_path)

print(f"Saved gene metadata: {var_path}")
print(f"Shape: {adata.var.shape}")
print(f"Columns: {list(adata.var.columns)}")

Saved gene metadata: ../data/bronze/osd352_brain/ingest_date=2026-03-12/var.parquet
Shape: (32285, 4)
Columns: ['gene_ids', 'feature_types', 'genome', 'interval']


In [17]:
# save sparse count matrix as HDF5
X_path = output_dir / 'X.h5'

with h5py.File(X_path, 'w') as f:
    # store as sparse CSR matrix components
    f.create_dataset('data', data=adata.X.data, compression='gzip')
    f.create_dataset('indices', data=adata.X.indices, compression='gzip')
    f.create_dataset('indptr', data=adata.X.indptr, compression='gzip')
    f.create_dataset('shape', data=adata.X.shape)

print(f"Saved count matrix: {X_path}")
print(f"Shape: {adata.X.shape}")

# calculate sparsity (percentage of zeros)
total_elements = adata.X.shape[0] * adata.X.shape[1]
non_zero_elements = adata.X.nnz
sparsity_percent = (1 - non_zero_elements / total_elements) * 100

print(f"sparsity: {sparsity_percent:.1f}% zeros")

Saved count matrix: ../data/bronze/osd352_brain/ingest_date=2026-03-12/X.h5
Shape: (32243, 32285)
sparsity: 96.0% zeros


In [15]:
# Calculate total size
files = ['obs.parquet', 'var.parquet', 'X.h5']
total_bytes = 0

for file in files:
    filepath = output_dir / file
    file_size = os.path.getsize(filepath)
    total_bytes += file_size

total_mb = total_bytes / (1024 * 1024)
print(f"\nTotal bronze layer size: {total_mb:.2f} MB")


Total bronze layer size: 95.56 MB


## Summary

**Bronze layer created successfully!**

**Output location:** `data/bronze/osd352_brain/ingest_date=2026-03-10/`

<span style='color:steelblue;'>**Files created:**</span>
- `obs.parquet` - Cell metadata (32,243 cells) with provenance
- `var.parquet` - Gene metadata (32,285 genes, unique names)
- `X.h5` - Sparse count matrix (CSR format, ~90% zeros)

<span style='color:steelblue;'>**Transformations applied:**</span>
- Made gene names unique (fixed Scanpy warning)
- Added provenance metadata (ingest_date, source_file, dataset_id, organism, tissue, technology, genome_build, processing_pipeline)
- Converted to Parquet (metadata) + HDF5 (matrix) format

<span style='color:steelblue;'>**Next steps:**</span>
- Silver layer: QC filtering and normalization
- Define QC thresholds based on exploration findings

#### <span style='color:tomato;'>Tasks Completed<small> in this notebook</small> </span>
- *Loaded raw data -* Read 10X HDF5 file (32,243 cells × 32,285 genes)
- *Fixed data quality issue -* Made gene names unique to resolve Scanpy warning
- *Added provenance tracking -* Attached 8 metadata fields to track data origin and processing
- *Created Hive-style partitioning -* Organized output by ingest_date for data lake compatibility
- *Separated metadata from matrix -* Split into obs.parquet (cells), var.parquet (genes), and X.h5 (counts)
- *Optimized storage -* Used Parquet for metadata (fast queries) and sparse HDF5 for matrix (space-efficient)
- *Verified outputs -* Confirmed all files created with size checks

**Result:** Raw data transformed into bronze layer following medallion architecture principles (minimal transformation + provenance).